# Cross-Encoder Grounder Evaluation

Evaluates the cross-encoder grounding model on:
1. **Original test domains**: search_and_rescue, warehouse, traffic_light
2. **LIBERO domains**: libero_10, libero_90, libero_object, libero_goal, libero_spatial

This tests both in-domain performance and cross-domain generalization.

In [ ]:
# Install dependencies
import sys, subprocess, pkgutil
def _pip(pkg): 
    if pkgutil.find_loader(pkg.split("==")[0]) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
_pip("transformers")
_pip("pandas")
_pip("scikit-learn")

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from collections import defaultdict
from dataclasses import dataclass

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report

# Disable warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ===========================================
# CONFIGURATION
# ===========================================

# Model
MODEL_DIR = Path("outputs_cross_encoder/final")

# Original test data (in-domain)
ORIGINAL_TEST_DIR = Path("grounding_data_combined/test")
ORIGINAL_DOMAINS = ["search_and_rescue", "traffic_light", "warehouse"]

# LIBERO test data (out-of-domain)
LIBERO_DATA_DIR = Path("libero_grounding_data")
LIBERO_SUITES = ["libero_10", "libero_90", "libero_object", "libero_goal", "libero_spatial"]

# Output
OUT_DIR = Path("cross_encoder_eval_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Whether to include domain/type context in candidate strings
INCLUDE_CONTEXT = True

# Batch size for scoring
BATCH_SIZE = 64

print(f"Model: {MODEL_DIR}")
print(f"Original test: {ORIGINAL_TEST_DIR}")
print(f"LIBERO data: {LIBERO_DATA_DIR}")
print(f"Output: {OUT_DIR}")

In [ ]:
# ===========================================
# MODEL DEFINITION
# ===========================================

class CrossEncoderForGrounding(nn.Module):
    """Cross-Encoder model for grounding."""
    
    def __init__(self, config):
        super().__init__()
        self.bert = AutoModel.from_config(config)
        self.classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(config.hidden_size, 1),
        )
    
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output).squeeze(-1)
        return {'logits': logits}


def load_model(model_dir: Path, device: torch.device):
    """Load cross-encoder model."""
    config = AutoConfig.from_pretrained(model_dir)
    model = CrossEncoderForGrounding(config)
    
    # Load weights
    weights_path = model_dir / "pytorch_model.bin"
    if not weights_path.exists():
        weights_path = model_dir / "model.safetensors"
    
    state_dict = torch.load(weights_path, map_location='cpu')
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    return model

In [ ]:
# ===========================================
# LOAD MODEL
# ===========================================
def load_model(model_dir: Path, device: torch.device):
    """Load cross-encoder model."""
    config = AutoConfig.from_pretrained(model_dir)
    model = CrossEncoderForGrounding(config)
    
    # Prefer safetensors if available
    safetensors_path = model_dir / "model.safetensors"
    pytorch_path = model_dir / "pytorch_model.bin"
    
    if safetensors_path.exists():
        from safetensors.torch import load_file
        state_dict = load_file(safetensors_path)
    elif pytorch_path.exists():
        state_dict = torch.load(pytorch_path, map_location='cpu', weights_only=False)
    else:
        raise FileNotFoundError(f"No model weights found in {model_dir}")
    
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    return model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("\nLoading model...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = load_model(MODEL_DIR, device)
print(f"Model loaded from: {MODEL_DIR}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ===========================================
# DATA LOADING UTILITIES
# ===========================================

def load_jsonl(path: Path) -> List[Dict]:
    """Load JSONL file."""
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def get_candidates_from_prefix(prefix: List[str]) -> List[str]:
    """Extract candidate tokens from prefix."""
    for marker in ['<const>', '<predicates>']:
        if marker in prefix:
            idx = prefix.index(marker)
            return prefix[idx + 1:]
    
    if len(prefix) > 4:
        return prefix[4:]
    return prefix


def get_grounding_context(prefix: List[str]) -> Tuple[str, str]:
    """Extract domain and type context from prefix."""
    domain = prefix[0] if prefix else ''
    
    type_context = ''
    if '<type>' in prefix:
        type_idx = prefix.index('<type>')
        if type_idx + 1 < len(prefix):
            type_context = prefix[type_idx + 1]
    elif '<types>' in prefix:
        type_context = 'predicate'
    
    return domain, type_context


def get_grounding_type_from_prefix(prefix: List[str]) -> str:
    """Infer grounding type from prefix."""
    if '<predicates>' in prefix:
        return 'predicate'
    elif '<const>' in prefix:
        return 'argument'
    return 'unknown'


def load_original_test_data(data_dir: Path, domains: List[str]) -> List[Dict]:
    """Load original test data."""
    all_data = []
    
    for domain in domains:
        path = data_dir / f"{domain}.jsonl"
        if not path.exists():
            print(f"  [warn] Not found: {path}")
            continue
        
        data = load_jsonl(path)
        # Add domain and grounding_type if not present
        for ex in data:
            if 'domain' not in ex:
                ex['domain'] = ex.get('prefix', [''])[0]
            if 'grounding_type' not in ex:
                ex['grounding_type'] = get_grounding_type_from_prefix(ex.get('prefix', []))
        
        print(f"  {domain}: {len(data)} examples")
        all_data.extend(data)
    
    return all_data


def load_libero_data(data_dir: Path, suites: List[str]) -> List[Dict]:
    """Load LIBERO grounding data."""
    all_data = []
    
    for suite in suites:
        path = data_dir / f"{suite}_grounding.jsonl"
        if not path.exists():
            print(f"  [warn] Not found: {path}")
            continue
        
        data = load_jsonl(path)
        for ex in data:
            ex['domain'] = ex.get('suite', suite)
        
        print(f"  {suite}: {len(data)} examples")
        all_data.extend(data)
    
    return all_data

In [ ]:
# ===========================================
# LOAD DATA
# ===========================================

print("Loading original test data...")
original_test_data = load_original_test_data(ORIGINAL_TEST_DIR, ORIGINAL_DOMAINS)
print(f"Total original test: {len(original_test_data)}")

print("\nLoading LIBERO data...")
libero_data = load_libero_data(LIBERO_DATA_DIR, LIBERO_SUITES)
print(f"Total LIBERO: {len(libero_data)}")

In [ ]:
# ===========================================
# SCORING UTILITIES
# ===========================================

def score_candidates(
    model,
    tokenizer,
    sentence: str,
    candidates: List[str],
    domain: str,
    type_context: str,
    device: torch.device,
    include_context: bool = True,
    batch_size: int = 64,
) -> Dict[str, float]:
    """Score all candidates for a sentence."""
    
    scores = {}
    
    # Prepare candidate strings
    if include_context:
        candidate_strs = [f"{domain} {type_context} : {c}" for c in candidates]
    else:
        candidate_strs = candidates
    
    # Batch scoring
    for i in range(0, len(candidates), batch_size):
        batch_candidates = candidate_strs[i:i + batch_size]
        batch_raw = candidates[i:i + batch_size]
        
        encoding = tokenizer(
            [sentence] * len(batch_candidates),
            batch_candidates,
            truncation=True,
            max_length=128,
            padding=True,
            return_tensors='pt',
        )
        encoding = {k: v.to(device) for k, v in encoding.items()}
        
        with torch.no_grad():
            outputs = model(**encoding)
            probs = torch.sigmoid(outputs['logits']).cpu().numpy()
        
        for j, raw_cand in enumerate(batch_raw):
            scores[raw_cand] = float(probs[j])
    
    return scores


def evaluate_dataset(
    model,
    tokenizer,
    data: List[Dict],
    device: torch.device,
    include_context: bool = True,
    batch_size: int = 64,
    dataset_name: str = "dataset",
) -> Tuple[Dict[str, Any], List[Dict]]:
    """Evaluate model on a dataset."""
    
    results = []
    
    print(f"\nEvaluating {dataset_name} ({len(data)} examples)...")
    
    for i, ex in enumerate(data):
        sentence = ex.get('sentence', [])
        if isinstance(sentence, list):
            sentence = ' '.join(sentence)
        
        targets = set(ex.get('target', []))
        prefix = ex.get('prefix', [])
        grounding_type = ex.get('grounding_type', 'unknown')
        domain = ex.get('domain', ex.get('suite', 'unknown'))
        
        domain_marker, type_context = get_grounding_context(prefix)
        candidates = get_candidates_from_prefix(prefix)
        
        if not targets or not candidates:
            continue
        
        # Score all candidates
        scores = score_candidates(
            model, tokenizer, sentence, candidates,
            domain_marker, type_context, device, include_context, batch_size
        )
        
        # Rank by score
        sorted_scores = sorted(scores.items(), key=lambda x: -x[1])
        prediction = sorted_scores[0][0] if sorted_scores else None
        
        # Get rank of correct answer
        rank = None
        for r, (cand, _) in enumerate(sorted_scores, 1):
            if cand in targets:
                rank = r
                break
        
        results.append({
            'id': ex.get('id', i),
            'domain': domain,
            'grounding_type': grounding_type,
            'sentence': sentence,
            'target': list(targets),
            'prediction': prediction,
            'correct': prediction in targets,
            'rank': rank,
            'top_5': sorted_scores[:5],
            'target_scores': {t: scores.get(t, 0.0) for t in targets},
            'num_candidates': len(candidates),
        })
        
        if (i + 1) % 500 == 0:
            acc_so_far = sum(1 for r in results if r['correct']) / len(results)
            print(f"  Processed {i + 1} / {len(data)} ({acc_so_far:.2%} acc)")
    
    # Compute metrics
    correct = sum(1 for r in results if r['correct'])
    total = len(results)
    
    mrr = sum(1.0 / r['rank'] for r in results if r['rank']) / total if total > 0 else 0
    at_3 = sum(1 for r in results if r['rank'] and r['rank'] <= 3)
    at_5 = sum(1 for r in results if r['rank'] and r['rank'] <= 5)
    
    metrics = {
        'accuracy': correct / total if total > 0 else 0,
        'accuracy_at_3': at_3 / total if total > 0 else 0,
        'accuracy_at_5': at_5 / total if total > 0 else 0,
        'mrr': mrr,
        'correct': correct,
        'total': total,
    }
    
    print(f"  Accuracy: {metrics['accuracy']:.4f} ({correct}/{total})")
    print(f"  MRR: {metrics['mrr']:.4f}")
    
    return metrics, results

In [ ]:
# ===========================================
# EVALUATE ON ORIGINAL TEST DATA (IN-DOMAIN)
# ===========================================

print("=" * 60)
print("ORIGINAL TEST DATA (IN-DOMAIN)")
print("=" * 60)

original_metrics, original_results = evaluate_dataset(
    model, tokenizer, original_test_data, device,
    include_context=INCLUDE_CONTEXT,
    batch_size=BATCH_SIZE,
    dataset_name="Original Test"
)

In [ ]:
# ===========================================
# EVALUATE ON LIBERO DATA (OUT-OF-DOMAIN)
# ===========================================

print("\n" + "=" * 60)
print("LIBERO DATA (OUT-OF-DOMAIN / GENERALIZATION)")
print("=" * 60)

libero_metrics, libero_results = evaluate_dataset(
    model, tokenizer, libero_data, device,
    include_context=INCLUDE_CONTEXT,
    batch_size=BATCH_SIZE,
    dataset_name="LIBERO"
)

In [ ]:
# ===========================================
# BREAKDOWN: ORIGINAL TEST DATA
# ===========================================

print("\n" + "=" * 60)
print("ORIGINAL TEST DATA - BREAKDOWN")
print("=" * 60)

# By domain
print("\n--- By Domain ---")
by_domain = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in original_results:
    d = r['domain']
    by_domain[d]['total'] += 1
    if r['correct']:
        by_domain[d]['correct'] += 1

domain_rows = []
for domain in sorted(by_domain.keys()):
    c = by_domain[domain]
    acc = c['correct'] / c['total'] if c['total'] > 0 else 0
    domain_rows.append({'domain': domain, 'accuracy': acc, 'correct': c['correct'], 'total': c['total']})
    print(f"  {domain}: {acc:.4f} ({c['correct']}/{c['total']})")

df_original_domain = pd.DataFrame(domain_rows)

# By grounding type
print("\n--- By Grounding Type ---")
by_type = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in original_results:
    gt = r['grounding_type']
    by_type[gt]['total'] += 1
    if r['correct']:
        by_type[gt]['correct'] += 1

type_rows = []
for gt in sorted(by_type.keys()):
    c = by_type[gt]
    acc = c['correct'] / c['total'] if c['total'] > 0 else 0
    type_rows.append({'grounding_type': gt, 'accuracy': acc, 'correct': c['correct'], 'total': c['total']})
    print(f"  {gt}: {acc:.4f} ({c['correct']}/{c['total']})")

df_original_type = pd.DataFrame(type_rows)

In [ ]:
# ===========================================
# BREAKDOWN: LIBERO DATA
# ===========================================

print("\n" + "=" * 60)
print("LIBERO DATA - BREAKDOWN")
print("=" * 60)

# By suite
print("\n--- By Suite ---")
by_suite = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in libero_results:
    s = r['domain']
    by_suite[s]['total'] += 1
    if r['correct']:
        by_suite[s]['correct'] += 1

suite_rows = []
for suite in sorted(by_suite.keys()):
    c = by_suite[suite]
    acc = c['correct'] / c['total'] if c['total'] > 0 else 0
    suite_rows.append({'suite': suite, 'accuracy': acc, 'correct': c['correct'], 'total': c['total']})
    print(f"  {suite}: {acc:.4f} ({c['correct']}/{c['total']})")

df_libero_suite = pd.DataFrame(suite_rows)

# By grounding type
print("\n--- By Grounding Type ---")
by_type_libero = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in libero_results:
    gt = r['grounding_type']
    by_type_libero[gt]['total'] += 1
    if r['correct']:
        by_type_libero[gt]['correct'] += 1

type_rows_libero = []
for gt in sorted(by_type_libero.keys()):
    c = by_type_libero[gt]
    acc = c['correct'] / c['total'] if c['total'] > 0 else 0
    type_rows_libero.append({'grounding_type': gt, 'accuracy': acc, 'correct': c['correct'], 'total': c['total']})
    print(f"  {gt}: {acc:.4f} ({c['correct']}/{c['total']})")

df_libero_type = pd.DataFrame(type_rows_libero)

# By predicate (for predicate grounding)
print("\n--- By Predicate ---")
by_pred = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in libero_results:
    if r['grounding_type'] == 'predicate':
        for t in r['target']:
            by_pred[t]['total'] += 1
            if r['correct']:
                by_pred[t]['correct'] += 1

pred_rows = []
for pred in sorted(by_pred.keys()):
    c = by_pred[pred]
    acc = c['correct'] / c['total'] if c['total'] > 0 else 0
    pred_rows.append({'predicate': pred, 'accuracy': acc, 'correct': c['correct'], 'total': c['total']})
    print(f"  {pred}: {acc:.4f} ({c['correct']}/{c['total']})")

df_libero_pred = pd.DataFrame(pred_rows)

In [ ]:
# ===========================================
# ERROR ANALYSIS
# ===========================================

print("\n" + "=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

# Original errors
original_errors = [r for r in original_results if not r['correct']]
print(f"\n--- Original Test Errors ---")
print(f"Total: {len(original_errors)} / {len(original_results)} ({len(original_errors)/len(original_results)*100:.1f}%)")

print("\nSample Errors:")
for e in original_errors[:5]:
    print(f"\n  [{e['grounding_type']}] {e['domain']}")
    print(f"    Sentence: {e['sentence'][:60]}...")
    print(f"    Target: {e['target']}")
    print(f"    Predicted: {e['prediction']}")
    print(f"    Rank: {e['rank']}")

# LIBERO errors
libero_errors = [r for r in libero_results if not r['correct']]
print(f"\n--- LIBERO Errors ---")
print(f"Total: {len(libero_errors)} / {len(libero_results)} ({len(libero_errors)/len(libero_results)*100:.1f}%)")

print("\nSample Errors:")
for e in libero_errors[:10]:
    print(f"\n  [{e['grounding_type']}] {e['domain']}")
    print(f"    Sentence: {e['sentence'][:60]}...")
    print(f"    Target: {e['target']}")
    print(f"    Predicted: {e['prediction']}")
    print(f"    Rank: {e['rank']} / {e['num_candidates']}")
    if len(e['top_5']) >= 3:
        print(f"    Top 3: {[(c, f'{s:.4f}') for c, s in e['top_5'][:3]]}")

In [ ]:
# ===========================================
# SAVE RESULTS
# ===========================================

print("\n" + "=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Custom JSON encoder for numpy types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# Save metrics
all_metrics = {
    'original_test': original_metrics,
    'libero': libero_metrics,
    'generalization_gap': original_metrics['accuracy'] - libero_metrics['accuracy'],
}
with open(OUT_DIR / "metrics.json", 'w') as f:
    json.dump(all_metrics, f, indent=2, cls=NumpyEncoder)
print(f"Saved: {OUT_DIR / 'metrics.json'}")

# Save breakdown CSVs
df_original_domain.to_csv(OUT_DIR / "original_by_domain.csv", index=False)
df_original_type.to_csv(OUT_DIR / "original_by_type.csv", index=False)
df_libero_suite.to_csv(OUT_DIR / "libero_by_suite.csv", index=False)
df_libero_type.to_csv(OUT_DIR / "libero_by_type.csv", index=False)
df_libero_pred.to_csv(OUT_DIR / "libero_by_predicate.csv", index=False)
print(f"Saved breakdown CSVs")

# Save detailed results
with open(OUT_DIR / "original_detailed.jsonl", 'w') as f:
    for r in original_results:
        f.write(json.dumps(r, cls=NumpyEncoder) + '\n')

with open(OUT_DIR / "libero_detailed.jsonl", 'w') as f:
    for r in libero_results:
        f.write(json.dumps(r, cls=NumpyEncoder) + '\n')
print(f"Saved detailed results")

# Save errors
with open(OUT_DIR / "original_errors.jsonl", 'w') as f:
    for e in original_errors:
        f.write(json.dumps(e, cls=NumpyEncoder) + '\n')

with open(OUT_DIR / "libero_errors.jsonl", 'w') as f:
    for e in libero_errors:
        f.write(json.dumps(e, cls=NumpyEncoder) + '\n')
print(f"Saved error files")

In [ ]:
# ===========================================
# FINAL SUMMARY
# ===========================================

print("\n" + "=" * 70)
print("CROSS-ENCODER EVALUATION SUMMARY")
print("=" * 70)

print(f"\n📊 MODEL: {MODEL_DIR}")

print(f"\n" + "-" * 50)
print(f"ORIGINAL TEST DATA (IN-DOMAIN)")
print(f"-" * 50)
print(f"  Accuracy:  {original_metrics['accuracy']:.4f} ({original_metrics['correct']}/{original_metrics['total']})")
print(f"  Acc@3:     {original_metrics['accuracy_at_3']:.4f}")
print(f"  MRR:       {original_metrics['mrr']:.4f}")
print(f"\n  By Domain:")
for _, row in df_original_domain.iterrows():
    print(f"    {row['domain']:25s}: {row['accuracy']:.4f}")

print(f"\n" + "-" * 50)
print(f"LIBERO DATA (OUT-OF-DOMAIN)")
print(f"-" * 50)
print(f"  Accuracy:  {libero_metrics['accuracy']:.4f} ({libero_metrics['correct']}/{libero_metrics['total']})")
print(f"  Acc@3:     {libero_metrics['accuracy_at_3']:.4f}")
print(f"  MRR:       {libero_metrics['mrr']:.4f}")
print(f"\n  By Suite:")
for _, row in df_libero_suite.iterrows():
    print(f"    {row['suite']:25s}: {row['accuracy']:.4f}")
print(f"\n  By Grounding Type:")
for _, row in df_libero_type.iterrows():
    print(f"    {row['grounding_type']:25s}: {row['accuracy']:.4f}")

print(f"\n" + "-" * 50)
print(f"GENERALIZATION")
print(f"-" * 50)
# gap = original_metrics['accuracy'] - libero_metrics['accuracy']
# print(f"  Original Acc:  {original_metrics['accuracy']:.4f}")
print(f"  LIBERO Acc:    {libero_metrics['accuracy']:.4f}")
# print(f"  Gap:           {gap:+.4f}")

print(f"\n📁 OUTPUT: {OUT_DIR}/")

In [ ]:
# ===========================================
# COMPARISON TABLE
# ===========================================

comparison_data = [
    {
        'Dataset': 'Original Test (In-Domain)',
        'Examples': original_metrics['total'],
        'Accuracy': f"{original_metrics['accuracy']:.2%}",
        'Acc@3': f"{original_metrics['accuracy_at_3']:.2%}",
        'MRR': f"{original_metrics['mrr']:.4f}",
    },
    {
        'Dataset': 'LIBERO (Out-of-Domain)',
        'Examples': libero_metrics['total'],
        'Accuracy': f"{libero_metrics['accuracy']:.2%}",
        'Acc@3': f"{libero_metrics['accuracy_at_3']:.2%}",
        'MRR': f"{libero_metrics['mrr']:.4f}",
    },
]

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "=" * 70)
print("COMPARISON TABLE")
print("=" * 70)
print(df_comparison.to_string(index=False))

df_comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'comparison.csv'}")